In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
with open("dataset_filter.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for item in data:
    rows.append({
        "username": item["authorMeta"]["name"],
        "followers": item["authorMeta"]["fans"],

        "create_time": pd.to_datetime(item["createTime"], unit="s"),
        "duration": item["videoMeta"]["duration"],

        "views": item["playCount"],
        "likes": item["diggCount"],
        "comments": item["commentCount"],
        "shares": item["shareCount"],

        "caption": item["text"],
        "hashtag": item["hashtags"],
        "musicOriginal": item['musicMeta']['musicOriginal'],
        "coverUrl": item['videoMeta']['coverUrl']
    })

df = pd.DataFrame(rows)
df = df.sort_values(by=["username", "create_time"], ascending=[True, True])

data = df

In [26]:
data.iloc[20],data.iloc[1]

(username                                              .sturdy.manz
 followers                                                     4489
 create_time                                    2025-09-06 21:37:14
 duration                                                        10
 views                                                         7077
 likes                                                          457
 comments                                                         7
 shares                                                          10
 caption                        wanna see u do a big twirl👀 #fypシ゚ 
 hashtag                                        [{'name': 'fypシ゚'}]
 musicOriginal                                                 True
 coverUrl         https://p16-common-sign.tiktokcdn-us.com/tos-a...
 Name: 13359, dtype: object,
 username                                              .sturdy.manz
 followers                                                     4489
 create_time       

In [17]:
import pandas as pd
import numpy as np

def build_bert_sequences(df, window_size=10):
    sequences = []
    weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    df = df.copy()
    
    df['hashtag_str'] = df['hashtag'].apply(
        lambda x: " ".join([tag['name'] for tag in x]) if isinstance(x, list) else ""
    )
    
    df['caption'] = df['caption'].fillna("").str.replace("\n", " ").str.strip()
    
    for username, group in df.groupby("username"):
        group = group.sort_values("create_time")
        if len(group) <= window_size:
            continue
            
        for i in range(len(group) - window_size):
            history = group.iloc[i : i + window_size]
            target = group.iloc[i + window_size]
            
            # ===== ACCOUNT =====
            f_count = target['followers']
            account_str = f"[ACCOUNT] followers={f_count}"
            
            # ===== TARGET =====
            t_day = weekdays[target['create_time'].weekday()]
            t_hour = target['create_time'].hour
            t_tags = target['hashtag_str']
            
            target_str = (
                "[TARGET] "
                f"day={t_day} hour={t_hour} "
                f"duration={target['duration']} "
                f"tags={t_tags if t_tags else 'none'} "
                f"caption=\"{target['caption'][:100]}\""
            )
            
            # ===== HISTORY =====
            history_segments = []
            for idx, (_, h_row) in enumerate(history.iterrows()):
                h_day = weekdays[h_row['create_time'].weekday()]
                h_hour = h_row['create_time'].hour
                h_tags = h_row['hashtag_str']
                
                h_str = (
                    f"[H{idx+1}] "
                    f"day={h_day} hour={h_hour} "
                    f"views={h_row['views']} "
                    f"likes={h_row['likes']} "
                    f"comments={h_row['comments']} "
                    f"shares={h_row['shares']} "
                    f"duration={h_row['duration']} "
                    f"tags={h_tags if h_tags else 'none'} "
                    f"caption=\"{h_row['caption'][:40]}\""
                )
                history_segments.append(h_str)
            
            full_text = "\n".join([account_str, target_str] + history_segments)
            
            # ===== LABEL =====
            explosion_score = target['views'] / (target['followers'])
            log_target = np.log1p(explosion_score)
            
            sequences.append({
                "username": username,
                "text": full_text,
                "label": float(log_target),
                "raw_views": target['views']
            })
            
    return pd.DataFrame(sequences)

bert_df = build_bert_sequences(df)

print(bert_df.iloc[0])

username                                          .sturdy.manz
text         [ACCOUNT] followers=4489\n[TARGET] day=Tue hou...
label                                                 1.205687
raw_views                                                10500
Name: 0, dtype: object


In [ ]:
bert_df['group_idx'] = bert_df.groupby('username').cumcount()

train_df = bert_df[bert_df['group_idx'] < 16].copy()

val_df = bert_df[(bert_df['group_idx'] >= 16) & (bert_df['group_idx'] < 18)].copy()

test_df = bert_df[(bert_df['group_idx'] >= 18) & (bert_df['group_idx'] < 20)].copy()

for df_split in [train_df, val_df, test_df]:
    df_split.drop(columns=['group_idx'], inplace=True)

print(f"训练集样本数: {len(train_df)} (每个用户 16 条)")
print(f"验证集样本数: {len(val_df)} (每个用户 2 条)")
print(f"测试集样本数: {len(test_df)} (每个用户 2 条)")

train_df.to_csv("train_bert.csv", index=False)
val_df.to_csv("val_bert.csv", index=False)
test_df.to_csv("test_bert.csv", index=False)

训练集样本数: 7456 (每个用户 16 条)
验证集样本数: 932 (每个用户 2 条)
测试集样本数: 932 (每个用户 2 条)
